# 52d. Journal figures 3, 4 and 6 re-exported at 600 dpi

The journal asks for figure files at 600 dpi. Three of the eight main figures are ROC panels, so they
cannot be redrawn from `reports/*.csv`: they need the cached per-record scores under `data/`. This
notebook reads those caches, redraws the three figures with the same content at 600 dpi, and gives the
legends their paper names instead of the configuration keys (`full_aug` becomes
`structural (paraphrase-augmented)`, and so on).

No model is loaded and no text is re-scored. If a cached score file is missing the stage stops and says
which one, rather than re-scoring.

**Outputs.** `figures/opi_second_source.png` (manuscript Figure 3), `figures/structural_detector_roc.png`
(Figure 4), `figures/agentdojo.png` (Figure 6), all at 600 dpi.

In [ ]:
# === SESSION BOOTSTRAP (run first): no GPU needed ===
!pip install -q transformers datasets sentencepiece scikit-learn
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
os.makedirs('figures', exist_ok=True)
import numpy as np, pandas as pd, json, importlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
DPI = 600
PAPER = {'structural (full)': 'structural (full)',
         'structural (full_aug)': 'structural (paraphrase-augmented)',
         'structural (ablation_no_twins)': 'structural (ablation, no twins)',
         'structural (full_sizematched)': 'structural (size-matched)'}
def nm(s): return PAPER.get(s, s)
print('Session ready. Re-export at', DPI, 'dpi; no GPU needed.')

## Stage A. Figure 3: the second source (notebook 19's panel)

In [ ]:
OPI = json.load(open('data/opi/opi_eval.json'))
R = OPI['records']; TXT = [r['text'] for r in R]; Y = np.array([r['label'] for r in R])
TAGS = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
SCORES = {}
missing = []
for t, lab in TAGS.items():
    fp = f'data/opi/score_{t}.npy'
    if os.path.exists(fp): SCORES[lab] = np.load(fp)
    else: missing.append(fp)
for cfg in ['full', 'ablation_no_twins', 'full_sizematched']:
    fp = f'data/opi/score_structural_{cfg}.npy'
    if os.path.exists(fp): SCORES[f'structural ({cfg})'] = np.load(fp)
    else: missing.append(fp)
assert not missing, 'missing cached scores:\n  ' + '\n  '.join(missing)
for k, v in SCORES.items(): assert len(v) == len(TXT), f'{k}: {len(v)} vs {len(TXT)}'
print('scorers:', [nm(k) for k in SCORES])

In [ ]:
BS = pd.read_csv('reports/opi_by_strategy.csv')
STRATEGIES = ['naive', 'escape', 'fake_completion', 'ignore', 'combined']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, p in SCORES.items():
    f, t, _ = roc_curve(Y, p)
    axes[0].plot(f, t, lw=2 if name == 'structural (full)' else 1.2, label=f'{nm(name)} ({roc_auc_score(Y, p):.2f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=.8); axes[0].axvline(0.01, color='grey', ls=':', lw=.8)
axes[0].set_xlabel('false-positive rate on matched benign data'); axes[0].set_ylabel('true-positive rate on injected data')
axes[0].set_title('Second source (Open-Prompt-Injection construction)'); axes[0].legend(fontsize=7, loc='lower right'); axes[0].grid(alpha=.3)
piv = BS.pivot_table(index='strategy', columns='model', values='FNR_guar').reindex(STRATEGIES)
piv.columns = [nm(c) for c in piv.columns]; assert len(piv)
piv.plot.bar(ax=axes[1], width=.8); axes[1].set_ylabel('FNR at guaranteed 1 percent FPR'); axes[1].set_title('By attack strategy')
axes[1].legend(fontsize=6); axes[1].grid(axis='y', alpha=.3); axes[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.savefig('figures/opi_second_source.png', dpi=DPI, bbox_inches='tight'); plt.show()
print('saved figures/opi_second_source.png at', DPI, 'dpi')

## Stage B. Figure 4: hijack injections against host documents (notebook 18's panel)

In [ ]:
import data_loaders; importlib.reload(data_loaders)
from data_loaders import load_bipia_categorized, BIPIA_MALICIOUS, BIPIA_HIJACK
bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS)
HIJ = bip.meta.isin(BIPIA_HIJACK).values; HOST = bip.label.values == 0
print('BIPIA rows:', len(bip), '| hijack:', int(HIJ.sum()), '| hosts:', int(HOST.sum()))

TAGS18 = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
S18 = {}; missing = []
for t, lab in TAGS18.items():
    fp = f'data/score_{t}_bipia_cat.npy'
    if os.path.exists(fp): S18[lab] = np.load(fp)
    else: missing.append(fp)
for cfg in ['full', 'ablation_no_twins', 'full_sizematched']:
    fp = f'data/structural/scores/{cfg}_bipia.npy'
    if os.path.exists(fp): S18[f'structural ({cfg})'] = np.load(fp)
    else: missing.append(fp)
if missing:
    print('NOT FOUND, check the names under data/ and data/structural/scores/:')
    for m in missing: print('   ', m)
    print('\nfiles present:'); print('   data/:', [f for f in os.listdir('data') if f.endswith('.npy')][:20])
    if os.path.isdir('data/structural/scores'): print('   data/structural/scores/:', sorted(os.listdir('data/structural/scores'))[:20])
    raise SystemExit('fill the paths above into S18 and re-run this cell')
for k, v in S18.items(): assert len(v) == len(bip), f'{k}: {len(v)} vs {len(bip)}'
print('scorers:', [nm(k) for k in S18])

In [ ]:
y = np.r_[np.ones(int(HIJ.sum())), np.zeros(int(HOST.sum()))]
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, p in S18.items():
    pp = np.r_[p[HIJ], p[HOST]]; f, t, _ = roc_curve(y, pp)
    ax.plot(f, t, lw=2 if name.startswith('structural (full)') else 1.2, label=f'{nm(name)} (AUROC {roc_auc_score(y, pp):.2f})')
ax.plot([0, 1], [0, 1], 'k--', lw=.8); ax.axvline(0.01, color='grey', ls=':', lw=.8)
ax.set_xlabel('false-positive rate on BIPIA host documents'); ax.set_ylabel('true-positive rate on hijack injections')
ax.set_title('Hijack injection versus benign host documents'); ax.legend(fontsize=7, loc='lower right'); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('figures/structural_detector_roc.png', dpi=DPI, bbox_inches='tight'); plt.show()
print('saved figures/structural_detector_roc.png at', DPI, 'dpi')

## Stage C. Figure 6: AgentDojo tool outputs (notebook 25's panel)

In [ ]:
RA = json.load(open('data/agentdojo/set.json'))
WINDOW_CHARS = 1600
KEEP = np.array([i for i, r in enumerate(RA) if r['label'] == 0 or r.get('inj_offset', 0) <= WINDOW_CHARS])
YA = np.array([RA[i]['label'] for i in KEEP])
LAB25 = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)',
         'deepset_injection': 'deepset injection', 'protectai_v1': 'ProtectAI-v1', 'fmops_distilbert': 'fmops distilbert',
         'piguard': 'PIGuard', 'conventional_ft': 'conventional fine-tune', 'tfidf': 'TF-IDF reference',
         'full': 'structural (full)', 'full_aug': 'structural (full_aug)'}
SA = {}
for tag, lab in LAB25.items():
    fp = f'data/agentdojo/score_{tag}.npy'
    if os.path.exists(fp): SA[lab] = np.load(fp)[KEEP]
assert SA, 'no cached AgentDojo scores found under data/agentdojo/'
print('scorers:', [nm(k) for k in SA])
print('records kept:', len(YA), '| injected:', int(YA.sum()))

In [ ]:
from agentdojo_source import ATTACKS
BA = pd.read_csv('reports/agentdojo_by_attack_strict.csv') if os.path.exists('reports/agentdojo_by_attack_strict.csv') else pd.read_csv('reports/agentdojo_by_attack.csv')
VAL = 'FNR_strict' if 'FNR_strict' in BA.columns else 'FNR_guar'
print('using', VAL, 'from', 'strict' if VAL == 'FNR_strict' else 'standard', 'file')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, p in SA.items():
    f, t, _ = roc_curve(YA, p); axes[0].plot(f, t, lw=2 if name == 'structural (full)' else 1.1, label=f'{nm(name)} ({roc_auc_score(YA, p):.2f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=.8); axes[0].axvline(0.01, color='grey', ls=':', lw=.8)
axes[0].set_xlabel('false-positive rate on benign tool outputs'); axes[0].set_ylabel('true-positive rate on injected tool outputs')
axes[0].set_title('AgentDojo tool outputs'); axes[0].legend(fontsize=6, loc='lower right'); axes[0].grid(alpha=.3)
piv = BA.pivot_table(index='attack', columns='model', values=VAL).reindex([a for a in ATTACKS if a in set(BA.attack)])
piv.columns = [nm(c) for c in piv.columns]; assert len(piv)
piv.plot.bar(ax=axes[1], width=.85); axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('FNR at guaranteed 1 percent FPR')
axes[1].set_title('By attack style'); axes[1].legend(fontsize=6, loc='upper left', bbox_to_anchor=(1.0, 1.0))
axes[1].grid(axis='y', alpha=.3); axes[1].tick_params(axis='x', labelrotation=30, labelsize=8)
plt.tight_layout(); plt.savefig('figures/agentdojo.png', dpi=DPI, bbox_inches='tight'); plt.show()
print('saved figures/agentdojo.png at', DPI, 'dpi')

## Stage D. Check and commit

In [ ]:
from PIL import Image
for fp in ['figures/opi_second_source.png', 'figures/structural_detector_roc.png', 'figures/agentdojo.png']:
    im = Image.open(fp); print(f'{fp}: {im.size[0]} x {im.size[1]} px, dpi {im.info.get("dpi")}')
    assert im.size[0] >= 3000, f'{fp} is still small: re-run the stage that writes it'
print('\nall three re-exported at 600 dpi')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb52d: re-export journal Figures 3, 4 and 6 at 600 dpi from the cached scores, with paper names in the legends"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)